In [3]:
from google.colab import files
import zipfile, os

uploaded = files.upload()  # pick rag_eval.zip
with zipfile.ZipFile("rag_eval.zip") as z:
    z.extractall("/content")
print(os.listdir("/content"))

Saving rag_eval.zip to rag_eval.zip
['.config', 'rag_eval', 'rag_eval.zip', 'sample_data']


In [15]:
!git clone https://github.com/HajarBraidi/financial-analyst /content/colleagues_repo

Cloning into '/content/colleagues_repo'...
remote: Enumerating objects: 49, done.
remote: Counting objects: 100% (49/49), done.
remote: Compressing objects: 100% (37/37), done.
remote: Total 49 (delta 15), reused 39 (delta 10), pack-reused 0 (from 0)
Receiving objects: 100% (49/49), 34.70 KiB | 4.96 MiB/s, done.
Resolving deltas: 100% (15/15), done.


In [16]:
!pip install -q langchain-qdrant langchain-huggingface sentence-transformers langchain-google-genai ragas pypdf python-dotenv

In [17]:
import os
from google.colab import userdata

for name in ["QDRANT_URL", "QDRANT_API_KEY", "QDRANT_COLLECTION", "GOOGLE_API_KEY", "GEMINI_MODEL"]:
    os.environ[name] = userdata.get(name)

In [18]:
!ls -la /content
!ls /content/colleagues_repo | head

total 463408
drwxr-xr-x 1 root root      4096 Oct 24 16:23 .
drwxr-xr-x 1 root root      4096 Oct 24 14:50 ..
drwxr-xr-x 4 root root      4096 Oct 24 16:23 colleagues_repo
drwxr-xr-x 4 root root      4096 Oct 21 13:23 .config
drwxr-xr-x 6 root root      4096 Oct 24 16:19 rag_eval
-rw-r--r-- 1 root root 474499855 Oct 24 16:19 rag_eval.zip
drwxr-xr-x 1 root root      4096 Oct 21 13:23 sample_data
indexation_colab.ipynb
indexation_kaggle.ipynb
README.md
requirements.txt
src


In [19]:
import os, shutil

PROJECT = "/content/rag_eval"
os.makedirs(PROJECT, exist_ok=True)

project_files = [
    "adapter.py", "eval_config.py", "build_golden_set.py", "retrieval_eval.py",
    "generation_eval.py", "run_eval.py", "analysis.py", "requirements.txt",
    ".env", ".gitignore", "data",
]
for name in project_files:
    src = os.path.join("/content", name)
    dst = os.path.join(PROJECT, name)
    if os.path.exists(src) and not os.path.exists(dst):
        shutil.move(src, dst)

print(sorted(os.listdir(PROJECT)))

['.env', '.gitignore', '.venv', '.vscode', '__pycache__', 'adapter.py', 'analysis.py', 'build_golden_set.py', 'data', 'eval_config.py', 'generation_eval.py', 'requirements.txt', 'retrieval_eval.py', 'run_eval.py']


In [20]:
import sys
sys.path.insert(0, "/content/rag_eval")
from eval_config import COLLEAGUES_SRC
print(COLLEAGUES_SRC, os.path.isdir(COLLEAGUES_SRC))

/content/colleagues_repo/src True


In [21]:
%cd /content/rag_eval
!python -c "from adapter import run_rag; r=run_rag('What was the FY2018 net sales for 3M?'); print(r['answer'][:300]); print(len(r['contexts']), 'contexts'); print(r['retrieved_ids'][:3])"

/content/rag_eval
Chargement de BAAI/bge-m3 sur cpu...
modules.json: 100% 349/349 [00:00<00:00, 871kB/s]
config_sentence_transformers.json: 100% 123/123 [00:00<00:00, 460kB/s]
README.md: 100% 15.8k/15.8k [00:00<00:00, 27.6MB/s]
sentence_bert_config.json: 100% 54.0/54.0 [00:00<00:00, 205kB/s]
config.json: 100% 687/687 [00:00<00:00, 2.46MB/s]

pytorch_model.bin: downloading bytes:  15% 336M/2.27G [00:03<00:23, 84.1MB/s, 27.0MB/s  ]
pytorch_model.bin: downloading bytes:  37% 838M/2.27G [00:06<00:10, 140MB/s, 57.0MB/s  ]
pytorch_model.bin: downloading bytes:  43% 980M/2.27G [00:10<00:19, 66.6MB/s, 53.0MB/s  ]
pytorch_model.bin: reconstructing file:  21% 469M/2.27G [00:19<00:33, 54.4MB/s, 33.1MB/s  ]
pytorch_model.bin: downloading bytes:  67% 1.52G/2.27G [00:26<00:03, 223MB/s, 50.6MB/s  ]
pytorch_model.bin: reconstructing file:  65% 1.48G/2.27G [00:28<00:15, 50.7MB/s, 68.4MB/s  ]
pytorch_model.bin: downloading bytes:  67% 1.52G/2.27G [00:39<00:03, 223MB/s, 51.8MB/s  ]
pytorch_model.bin: dow

In [34]:
%cd /content/rag_eval
!python build_golden_set.py
!wc -l data/golden_set.jsonl

/content/rag_eval
Loaded 90456 chunks from 'financebench'
/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3919: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3919: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3919: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3919: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) tem

In [33]:
for name in ["QDRANT_URL", "QDRANT_API_KEY", "QDRANT_COLLECTION", "GOOGLE_API_KEY", "GEMINI_MODEL", "GEMINI_QUESTION_MODEL"]:
    os.environ[name] = userdata.get(name)

In [35]:
import json

PATH = "data/golden_set.jsonl"
DROP = {"q009"}

rows = [json.loads(l) for l in open(PATH, encoding="utf-8") if l.strip()]
kept = []
for r in rows:
    if r["id"] in DROP:
        continue
    r["reviewed"] = True
    kept.append(r)

with open(PATH, "w", encoding="utf-8") as f:
    for r in kept:
        f.write(json.dumps(r, ensure_ascii=False) + "\n")

print(len(kept), "questions kept")

14 questions kept


In [36]:
%cd /content/rag_eval
!python retrieval_eval.py

/content/rag_eval
Evaluating 14 reviewed questions, k=8
Chargement de BAAI/bge-m3 sur cpu...
Loading weights: 100% 391/391 [00:00<00:00, 13672.59it/s]

=== Retrieval summary ===
hit_rate     0.786
recall       0.786
precision    0.098
MRR          0.589

=== By question type ===
               hit_rate  recall  precision     rr
question_type                                    
factual           0.786   0.786      0.098  0.589


In [37]:
from google.colab import files
files.download("/content/rag_eval/data/golden_set.jsonl")
files.download("/content/rag_eval/data/results/retrieval_per_question.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>